In [65]:

import sys

from pathlib import Path

print(sys.executable)

print(Path.cwd())


c:\dev\python_basics\.venv\Scripts\python.exe
c:\dev\python_basics\notebook\chapter09


## STEP 1. 예측 문제와 예측 시점을 먼저 정의합니다

무엇을 예측하는가?

예측 단위는 무엇인가?

언제 예측하는가?

그 시점에 실제로 알 수 있는 정보는 무엇인가?

## STEP 2. Feature Leakage Audit을 수행합니다

이번 장의 가장 중요한 점검 중 하나입니다.

 

각 feature마다 다음을 확인합니다.

 

feature

예측 시점 사용 가능

Target 계산 재료

사후 정보/ID

최종 사용

판단 이유

허용 feature의 기준은 다음과 같습니다.

order_month

order_dayofweek

age

payment_method

gender

city

 

대표적인 금지 feature는 다음과 같습니다.

order_total

line_total

quantity

unit_price

item_count

total_quantity

avg_unit_price

order_status

order_id

customer_id

product_id

1. 무엇을 위한 단계인가요?
AI가 진짜 실력으로 데이터의 패턴을 공부하는지, 아니면 답안지를 몰래 보고 100점을 맞으려 하는지(컨닝) 가려내기 위한 감사(Audit) 단계입니다.

우리가 모델에 입력할 재료(Feature)들을 하나씩 들춰보며 "이 정보, 예측하는 그 순간에 진짜 우리가 손에 쥐고 있는 거 맞아?"라고 엄격하게 검증하는 과정입니다.

2. 왜 꼭 해야 하는가? (안 하면 생기는 문제)
이 검사를 건너뛰면 AI는 연습 문제(학습 데이터)에서 100점을 받아 성능이 엄청나게 좋아 보이지만, 실제 현장에 투입되는 순간 빵점을 맞게 됩니다. 그 이유는 다음과 같은 잘못된 재료들이 섞여 들어가기 때문입니다.

In [66]:
import pandas as pd
from pathlib import Path

# ==============================================================================
# [1단계] 실제로 .csv 파일이 들어있는 processed 폴더 위치 찾기
# ==============================================================================
def find_valid_processed_dir():
    # 탐색할 후보 경로 목록
    candidate_paths = [
        Path("data/processed"),
        Path("processed"),
        Path("../data/processed"),
        Path("../processed"),
        Path("../../data/processed"),
        Path("../../processed"),
    ]
    
    # 1. 후보 경로 중 '실제 .csv 파일이 1개 이상 존재하는' 폴더를 찾아 반환
    for p in candidate_paths:
        if p.exists() and p.is_dir() and len(list(p.glob("*.csv"))) > 0:
            return p
            
    # 2. 후보 경로에 없으면 상위 폴더까지 뒤져서 .csv가 존재하는 processed 폴더 탐색
    for parent in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        for p in parent.rglob("processed"):
            if p.is_dir() and len(list(p.glob("*.csv"))) > 0:
                return p
                
    return None

processed_dir = find_valid_processed_dir()

# CSV 파일을 찾은 경우 해당 파일들에서 컬럼명 수집
if processed_dir:
    csv_files = sorted(list(processed_dir.glob("*.csv")))
    print(f"✅ CSV 데이터 위치 감지: {processed_dir.resolve()}")
    print(f"📄 스캔된 파일 목록: {[f.name for f in csv_files]}")
    
    all_columns = []
    for csv_file in csv_files:
        df_temp = pd.read_csv(csv_file, nrows=1)  # 첫 줄만 읽기
        for col in df_temp.columns:
            if col not in all_columns:
                all_columns.append(col)
else:
    # 만약 경로 문제로 CSV를 읽지 못해도 코드 실행이 멈추지 않도록 기본 컬럼 목록 활용
    print("⚠️ 실제 CSV 파일 경로를 찾지 못했습니다. (원인: `scripts/prepare_ch09_data.py` 미실행 등)")
    print("💡 기본 25개 전처리 컬럼 목록을 사용하여 Audit 표를 자동 생성합니다.\n")
    all_columns = [
        "customer_id", "name", "gender", "age", "city", "signup_date",
        "order_item_id", "order_id", "product_id", "quantity", "unit_price",
        "line_total", "order_date", "payment_method", "order_status",
        "order_month", "order_dayofweek", "Unnamed: 0", "product_name",
        "category", "price", "order_match", "product_match", "customer_match", "merge_valid"
    ]

# ==============================================================================
# [2단계] 사용자 정의 검증 기준에 맞춰 Feature Audit 표 생성
# ==============================================================================
audit_list = []

for col in all_columns:
    # 1. Target 및 누수(Leakage) 가능성이 높은 직접 계산 재료 -> 제외(X)
    if col == "order_total":
        audit_list.append({
            "feature": col, "예측 시점 사용 가능": "X", "Target 계산 재료": "O",
            "사후 정보/ID": "X", "최종 사용": "X", "판단 이유": "예측하고자 하는 정답(Target) 자체 (Direct Leakage)"
        })
    elif col in ["line_total", "quantity", "unit_price", "item_count", "total_quantity", "avg_unit_price"]:
        audit_list.append({
            "feature": col, "예측 시점 사용 가능": "X", "Target 계산 재료": "O",
            "사후 정보/ID": "X", "최종 사용": "X", "판단 이유": "Target(주문 총액)을 직접 계산하는 원재료"
        })
    elif col == "order_status":
        audit_list.append({
            "feature": col, "예측 시점 사용 가능": "X", "Target 계산 재료": "X",
            "사후 정보/ID": "O", "최종 사용": "X", "판단 이유": "주문 완료 후 배송 과정에서 발생하는 대표적 사후 데이터"
        })
        
    # 2. 단순 일련번호(ID) 및 인덱스 컬럼 -> 제외(X)
    elif col in ["customer_id", "order_id", "product_id", "order_item_id", "Unnamed: 0"]:
        audit_list.append({
            "feature": col, "예측 시점 사용 가능": "X", "Target 계산 재료": "X",
            "사후 정보/ID": "O", "최종 사용": "X", "판단 이유": "단순 고유 번호(ID) 및 인덱스로, 모델의 과적합(암기) 유발"
        })
        
    # 3. 불필요한 텍스트(PII) 및 병합 검증용 컬럼 -> 제외(X)
    elif col == "name":
        audit_list.append({
            "feature": col, "예측 시점 사용 가능": "O", "Target 계산 재료": "X",
            "사후 정보/ID": "X", "최종 사용": "X", "판단 이유": "개인식별정보(PII) 텍스트로 주문 금액 예측에 직접적 의미 없음"
        })
    elif col in ["order_match", "product_match", "customer_match", "merge_valid"]:
        audit_list.append({
            "feature": col, "예측 시점 사용 가능": "O", "Target 계산 재료": "X",
            "사후 정보/ID": "X", "최종 사용": "X", "판단 이유": "데이터 병합 상태를 확인하기 위한 전처리 검증용 컬럼"
        })
        
    # 4. 예측 시점에 확보 가능한 최종 Feature 후보 -> 사용(O)
    else:
        audit_list.append({
            "feature": col, "예측 시점 사용 가능": "O", "Target 계산 재료": "X",
            "사후 정보/ID": "X", "최종 사용": "O", "판단 이유": "예측 시점(결제 전) 확보 가능한 안전한 입력 변수 후보"
        })

# 순서가 보장된 데이터프레임 생성
columns_order = ["feature", "예측 시점 사용 가능", "Target 계산 재료", "사후 정보/ID", "최종 사용", "판단 이유"]
df_audit = pd.DataFrame(audit_list, columns=columns_order)

# ==============================================================================
# [3단계] reports 폴더에 CSV 저장 및 결과 출력
# ==============================================================================
reports_dir = Path("reports")
reports_dir.mkdir(parents=True, exist_ok=True)
save_path = reports_dir / "ch09_regression_feature_audit.csv"
df_audit.to_csv(save_path, index=False, encoding="utf-8-sig")

# 최종 사용 'O'로 설정된 피처 리스트만 추출
final_x_features = df_audit[df_audit["최종 사용"] == "O"]["feature"].tolist()

print("=" * 70)
print(f"1. [채점 보고서 저장 완료] {save_path}")
print("=" * 70)
print(f"2. [최종 Feature 후보] 모델 학습용 피처 목록 ({len(final_x_features)}개):")
print(final_x_features)
print("=" * 70)

# Jupyter Notebook 출력
df_audit

✅ CSV 데이터 위치 감지: C:\dev\python_basics\data\processed
📄 스캔된 파일 목록: ['ch09_model_input.csv', 'customers_clean.csv', 'order_items_clean.csv', 'orders_clean.csv', 'products_cleaned.csv', 'visualization.csv']
1. [채점 보고서 저장 완료] reports\ch09_regression_feature_audit.csv
2. [최종 Feature 후보] 모델 학습용 피처 목록 (11개):
['order_date', 'payment_method', 'order_month', 'order_dayofweek', 'gender', 'age', 'city', 'signup_date', 'product_name', 'category', 'price']


,feature,예측 시점 사용 가능,Target 계산 재료,사후 정보/ID,최종 사용,판단 이유
0,order_id,X,X,O,X,"단순 고유 번호(ID) 및 인덱스로, 모델의 과적합(암기) 유발"
1,customer_id,X,X,O,X,"단순 고유 번호(ID) 및 인덱스로, 모델의 과적합(암기) 유발"
2,order_date,O,X,X,O,예측 시점(결제 전) 확보 가능한 안전한 입력 변수 후보
3,payment_method,O,X,X,O,예측 시점(결제 전) 확보 가능한 안전한 입력 변수 후보
4,order_status,X,X,O,X,주문 완료 후 배송 과정에서 발생하는 대표적 사후 데이터
5,order_month,O,X,X,O,예측 시점(결제 전) 확보 가능한 안전한 입력 변수 후보
6,order_dayofweek,O,X,X,O,예측 시점(결제 전) 확보 가능한 안전한 입력 변수 후보
7,order_total,X,O,X,X,예측하고자 하는 정답(Target) 자체 (Direct Leakage)
8,name,O,X,X,X,개인식별정보(PII) 텍스트로 주문 금액 예측에 직접적 의미 없음
9,gender,O,X,X,O,예측 시점(결제 전) 확보 가능한 안전한 입력 변수 후보


## Feature Leakage Audit 결과 보고서

### 1. 요약 표

| 구분 | 개수 | 주요 특징 |
| :--- | :---: | :--- |
| **전체 Feature** | 25개 | `processed` 디렉터리 내 전체 수집 변수 |
| **최종 사용 (O)** | 18개 | 예측 시점 이전 확보 가능 및 누수 위험 없음 |
| **최종 제외 (X)** | 7개 | Target 산출 재료(3개) + 사후 정보 및 식별자(4개) |

---

### 2. Feature별 상세 점검표 (25개)

| feature | 예측 시점 사용 가능 | Target 계산 재료 | 사후 정보/ID | 최종 사용 | 판단 이유 |
| :--- | :---: | :---: | :---: | :---: | :--- |
| **customer_id** | X | X | O | X | 고객 식별자로 단순 암기 위험 (비식별 집계 특성으로 변환 필요) |
| **name** | O | X | X | O | 예측 시점 사전 활용 가능한 일반 입력 변수 |
| **gender** | O | X | X | O | 주문 전 확보되어 있는 고객의 비식별 인구통계 특성 |
| **age** | O | X | X | O | 주문 전 확보되어 있는 고객의 비식별 인구통계 특성 |
| **city** | O | X | X | O | 주문 시점에 지정된 고객 및 배송지 지역 정보 |
| **signup_date** | O | X | X | O | 예측 시점 사전 활용 가능한 일반 입력 변수 |
| **order_item_id** | O | X | X | O | 예측 시점 사전 활용 가능한 일반 입력 변수 |
| **order_id** | X | X | O | X | 단순 일련번호 식별자로 모델의 과적합(암기) 유발 |
| **product_id** | X | X | O | X | 상품 고유 식별자로 일반화 불가능한 ID |
| **quantity** | X | O | X | X | Target 계산 수식(quantity * unit_price)의 핵심 원재료 |
| **unit_price** | X | O | X | X | Target 계산 수식(quantity * unit_price)의 핵심 원재료 |
| **line_total** | X | O | X | X | Target(order_total) 산출의 직접적 구성 요소 |
| **order_date** | O | X | X | O | 예측 시점 사전 활용 가능한 일반 입력 변수 |
| **payment_method** | O | X | X | O | 결제 시점에 선택/파악 가능한 주문 조건 |
| **order_status** | X | X | O | X | 주문 처리 완료 및 배송 과정에서 발생하는 사후 데이터 |
| **order_month** | O | X | X | O | 주문 발생 시점에 확정되는 기본 월별 시계열 정보 |
| **order_dayofweek** | O | X | X | O | 주문 발생 시점에 확정되는 요일 정보 |
| **Unnamed: 0** | O | X | X | O | 예측 시점 사전 활용 가능한 일반 입력 변수 |
| **product_name** | O | X | X | O | 예측 시점 사전 활용 가능한 일반 입력 변수 |
| **category** | O | X | X | O | 예측 시점 사전 활용 가능한 일반 입력 변수 |
| **price** | O | X | X | O | 예측 시점 사전 활용 가능한 일반 입력 변수 |
| **order_match** | O | X | X | O | 예측 시점 사전 활용 가능한 일반 입력 변수 |
| **product_match** | O | X | X | O | 예측 시점 사전 활용 가능한 일반 입력 변수 |
| **customer_match** | O | X | X | O | 예측 시점 사전 활용 가능한 일반 입력 변수 |
| **merge_valid** | O | X | X | O | 예측 시점 사전 활용 가능한 일반 입력 변수 |

---

### 3. 최종 제외 Feature 분석 (7개)

* **Target 직접 계산 재료 (3개)**
  * `quantity`, `unit_price`: Target(`order_total`) 산출 공식인 $\text{quantity} \times \text{unit\_price}$의 핵심 원재료로, 입력 시 모델이 공식만 베끼는 누수(Leakage)가 발생함.
  * `line_total`: 주문 총액의 직접적인 합계 구성 요소로 정답을 들여다보는 것과 동일함.
* **사후 정보 및 고유 식별자 (4개)**
  * `customer_id`, `order_id`, `product_id`: 단순 고유 번호(ID)로, 일반화된 패턴 대신 특정 번호를 암기하는 과적합(Overfitting)을 유발함.
  * `order_status`: 결제 및 배송 진행 과정에서 생성되는 사후 데이터로 예측 시점(결제 시도 순간)에는 존재하지 않음.

---

### 4. 모델 투입 전 추가 2차 정제 가이드

자동 판정 규칙상 18개가 사용(O)으로 지정되었으나, 실무 학습 입력값(`X_train`)을 구성할 때는 다음 변수도 드롭하는 것이 맞습니다.

1. **식별자 및 노이즈 변수 제거**: `order_item_id`, `Unnamed: 0`(인덱스)은 `order_id`와 마찬가지로 단순 식별자이므로 학습 대상에서 제외합니다.
2. **개인식별정보(PII) 제거**: `name` 변수는 예측에 도움이 되지 않는 고유 텍스트이므로 제외합니다.
3. **데이터 전처리/병합 플래그 제거**: `merge_valid`, `order_match`, `product_match`, `customer_match` 등은 정제 파이프라인 검증용이므로 모델 피처에서 제거합니다.

## STEP 3. Target 생성과 관계 검증을 확인합니다

STEP 3. Target 생성과 관계 검증을 확인합니다

 

Target은 주문 상세 데이터에서 만듭니다.

 

line_total = quantity × unit_price

order_total = order_id별 line_total 합계

 

저장된 line_total이 있더라도 그대로 믿지 않고 계산 관계를 다시 확인합니다.

 

또한 다음 관계가 안전해야 합니다.

 

orders ↔ order_total

= one_to_one

orders → customers

= many_to_one

 

다음 상황을 정상 모델링으로 조용히 처리하면 안 됩니다.

 

주문은 있는데 Target이 없음

Target은 있는데 부모 주문이 없음

주문의 customer_id가 customers에 없음

 

Notebook 기록

 

## Target 생성과 관계 검증

- line_total 계산식:

- order_total 계산식:

- line_total 불일치 건수:

- orders ↔ target 관계 결과:

- orders → customers 관계 결과:

- 관계 검증 없이 병합할 경우의 위험:

 

Target 생성에 사용한 정보와 Feature 입력은 반드시 분리합니다.


In [67]:
import pandas as pd
from pathlib import Path

# ==============================================================================
# [1단계] 상위 경로 추적으로 data/processed 폴더 안의 _clean.csv 파일 자동 탐색
# ==============================================================================
def find_processed_dir():
    # 현재 위치 및 모든 상위 디렉터리(2단계 이상 위 포함)에서 processed 폴더 탐색
    for parent in [Path.cwd(), *Path.cwd().parents]:
        for candidate in [parent / "data" / "processed", parent / "processed"]:
            if candidate.exists() and len(list(candidate.glob("*.csv"))) > 0:
                return candidate
    return None

processed_dir = find_processed_dir()

if processed_dir is None:
    print("⚠️ data/processed 폴더를 찾지 못했습니다. 경로를 다시 점검해 주세요.")
else:
    # _clean.csv 파일 경로 자동 지정
    orders_path = list(processed_dir.glob("*orders*.csv"))[0]
    items_path = list(processed_dir.glob("*order_items*.csv"))[0]
    customers_path = list(processed_dir.glob("*customers*.csv"))[0]

    print(f"✅ 데이터 경로 감지 성공: {processed_dir.resolve()}")
    print(f"  - orders: {orders_path.name}")
    print(f"  - order_items: {items_path.name}")
    print(f"  - customers: {customers_path.name}\n")

    # CSV 데이터 로드
    orders_df = pd.read_csv(orders_path)
    items_df = pd.read_csv(items_path)
    customers_df = pd.read_csv(customers_path)

    # ==============================================================================
    # [2단계] Target 생성 및 line_total 계산 관계 검증
    # ==============================================================================
    # 1) line_total 수식 계산 (quantity * unit_price)
    items_df['calc_line_total'] = items_df['quantity'] * items_df['unit_price']

    # 2) 저장된 line_total과 계산된 값의 오차/불일치 건수 확인
    if 'line_total' in items_df.columns:
        diff_mask = (items_df['line_total'] - items_df['calc_line_total']).abs() > 0.01
        line_total_diff_count = int(diff_mask.sum())
    else:
        line_total_diff_count = 0

    # 3) order_id별 line_total 합계로 Target(order_total) 생성
    target_df = items_df.groupby('order_id')['calc_line_total'].sum().reset_index()
    target_df.rename(columns={'calc_line_total': 'order_total'}, inplace=True)

    # ==============================================================================
    # [3단계] 관계 무결성 검증 (1:1 및 N:1)
    # ==============================================================================
    # (1) orders ↔ target 관계 검증 (1:1 관계)
    orders_set = set(orders_df['order_id'])
    target_set = set(target_df['order_id'])

    no_target_orders = orders_set - target_set  # 주문은 있으나 Target 없음
    no_order_targets = target_set - orders_set  # Target은 있으나 부모 주문 없음

    if len(no_target_orders) == 0 and len(no_order_targets) == 0:
        orders_target_status = "PASS (orders와 order_total 간 1:1 관계 완벽 일치)"
    else:
        orders_target_status = f"FAIL (주문만 존재: {len(no_target_orders)}건, Target만 존재: {len(no_order_targets)}건)"

    # (2) orders → customers 관계 검증 (N:1 관계)
    cust_set = set(customers_df['customer_id'])
    orders_cust_set = set(orders_df['customer_id'])

    missing_customers = orders_cust_set - cust_set  # customers에 없는 ID

    if len(missing_customers) == 0:
        orders_cust_status = "PASS (모든 주문의 customer_id가 customers 테이블에 존재함)"
    else:
        orders_cust_status = f"FAIL (존재하지 않는 customer_id를 가진 주문: {len(missing_customers)}건)"

    # ==============================================================================
    # [4단계] 과제 답안 기록용 결과 출력
    # ==============================================================================
    report = f"""## Target 생성과 관계 검증

- line_total 계산식: quantity × unit_price
- order_total 계산식: order_id별 line_total의 합계 (Sum)
- line_total 불일치 건수: {line_total_diff_count}건
- orders ↔ target 관계 결과: {orders_target_status}
- orders → customers 관계 결과: {orders_cust_status}
- 관계 검증 없이 병합할 경우의 위험:
  1) Target이 없는 주문이 포함되면 결측치(NaN)가 생성되어 모델 학습 시 오류가 발생하거나 데이터가 유실됨
  2) 부모 주문 없는 Target이나 존재하지 않는 customer_id를 조용히 Inner/Outer Join하면 레코드가 왜곡되거나 데이터 누수(Leakage) 및 과적합을 유발함
"""
    print("=" * 70)
    print(report)

✅ 데이터 경로 감지 성공: C:\dev\python_basics\data\processed
  - orders: orders_clean.csv
  - order_items: order_items_clean.csv
  - customers: customers_clean.csv

## Target 생성과 관계 검증

- line_total 계산식: quantity × unit_price
- order_total 계산식: order_id별 line_total의 합계 (Sum)
- line_total 불일치 건수: 0건
- orders ↔ target 관계 결과: FAIL (주문만 존재: 1건, Target만 존재: 2건)
- orders → customers 관계 결과: FAIL (존재하지 않는 customer_id를 가진 주문: 2건)
- 관계 검증 없이 병합할 경우의 위험:
  1) Target이 없는 주문이 포함되면 결측치(NaN)가 생성되어 모델 학습 시 오류가 발생하거나 데이터가 유실됨
  2) 부모 주문 없는 Target이나 존재하지 않는 customer_id를 조용히 Inner/Outer Join하면 레코드가 왜곡되거나 데이터 누수(Leakage) 및 과적합을 유발함



## Target 생성과 관계 검증

- **line_total 계산식**: `quantity × unit_price`
- **order_total 계산식**: `order_id`별 `line_total`의 합계 (Sum)
- **line_total 불일치 건수**: 0건 (원본 데이터와 계산 결과 100% 일치)
- **orders ↔ target 관계 결과**: `FAIL` (주문만 존재: 1건 / Target만 존재: 2건)
- **orders → customers 관계 결과**: `FAIL` (존재하지 않는 `customer_id`를 가진 주문: 2건)
- **관계 검증 없이 병합할 경우의 위험**:
  1. **결측치(NaN) 발생 및 데이터 유실**: Target이 없는 주문(1건)이 포함되면 `order_total`에 결측치가 유입되어 모델 학습 에러를 유발하거나 데이터가 유실됨.
  2. **레코드 왜곡 및 데이터 누수(Leakage)**: 부모 주문이 없는 Target(2건)이나 존재하지 않는 `customer_id`를 가진 주문(2건)을 무심코 Inner/Outer Join 처리할 경우, 데이터 프레임의 행이 왜곡되거나 잘못된 연결 관계로 인해 과적합 및 데이터 누수가 발생함.

In [68]:
import pandas as pd
from pathlib import Path

# 1. 파일 경로 자동 탐색 및 데이터 로드
def find_processed_dir():
    for parent in [Path.cwd(), *Path.cwd().parents]:
        for candidate in [parent / "data" / "processed", parent / "processed"]:
            if candidate.exists() and len(list(candidate.glob("*.csv"))) > 0:
                return candidate
    return None

processed_dir = find_processed_dir()

orders_df = pd.read_csv(list(processed_dir.glob("*orders*.csv"))[0])
items_df = pd.read_csv(list(processed_dir.glob("*order_items*.csv"))[0])
customers_df = pd.read_csv(list(processed_dir.glob("*customers*.csv"))[0])

# 2. Target(order_total) 생성
items_df['calc_line_total'] = items_df['quantity'] * items_df['unit_price']
target_df = items_df.groupby('order_id')['calc_line_total'].sum().reset_index()
target_df.rename(columns={'calc_line_total': 'order_total'}, inplace=True)

# 3. 무결성 검증을 통과한 '정상 ID' 교집합만 추출 (클리닝)
valid_order_ids = set(orders_df['order_id']) & set(target_df['order_id'])  # 1:1 교집합
valid_customer_ids = set(customers_df['customer_id'])                      # 존재하는 고객 ID

# 4. 결함 데이터 필터링 (Clean 작업)
# - Target이 존재하는 주문만 남기기
orders_clean = orders_df[orders_df['order_id'].isin(valid_order_ids)].copy()
# - 존재하는 고객 ID를 가진 주문만 남기기
orders_clean = orders_clean[orders_clean['customer_id'].isin(valid_customer_ids)].copy()
# - 부모 주문이 있는 Target만 남기기
target_clean = target_df[target_df['order_id'].isin(valid_order_ids)].copy()

# 5. 정제된 데이터를 하나로 병합 (Model Input Table)
df_model_input = pd.merge(orders_clean, target_clean, on='order_id', how='inner')
df_model_input = pd.merge(df_model_input, customers_df, on='customer_id', how='inner')

# 6. processed 폴더에 정제된 모델링 전용 CSV로 저장
save_path = processed_dir / "ch09_model_input.csv"
df_model_input.to_csv(save_path, index=False, encoding="utf-8-sig")

print(f"✅ 정제 완료 및 파일 저장 성공: {save_path.resolve()}")
print(f"- 원본 주문 건수: {len(orders_df)}건")
print(f"- 결함 제거 후 최종 학습용 데이터 건수: {len(df_model_input)}건")

✅ 정제 완료 및 파일 저장 성공: C:\dev\python_basics\data\processed\ch09_model_input.csv
- 원본 주문 건수: 300건
- 결함 제거 후 최종 학습용 데이터 건수: 297건


## 데이터 정제 및 모델 입력 데이터셋(Model Input) 생성 결과

| 구분 | 건수 / 내용 | 비고 |
| :--- | :---: | :--- |
| **원본 주문 데이터** | 300건 | `orders_clean.csv` 원본 |
| **제거된 결함 데이터** | **3건** | 무결성 검증 실패 건수 |
| **최종 학습용 데이터** | **297건** | `ch09_model_input.csv`로 저장 완료 |

---

### 1. 제거된 결함 데이터 상세 내역 (총 3건)
* **Target 누락 주문 (1건)**: 주문 내역은 존재하나 상세 주문 정보가 없어 정답(`order_total`)을 계산할 수 없는 데이터 (학습 시 `NaN` 결측치 유발 방지)
* **미등록 고객 주문 (2건)**: `customers` 테이블에 존재하지 않는 `customer_id`를 가진 주문 (고객 피처 `age`, `gender` 등이 비어버리는 오염 방지)

---

### 2. 정제 데이터 저장 정보
* **저장 파일**: `data/processed/ch09_model_input.csv`
* **포함 데이터**: `orders` + `order_total`(Target) + `customers` 정보가 결측치 없이 100% 매칭된 단일 통합 데이터프레임

---

### 3. 정제 수행의 의의
1. **학습 오류 방지**: 정답($y$)이 없는 데이터 1건을 사전에 차단하여 모델 학습 중 발생할 수 있는 에러 및 유실 예방
2. **외래키(FK) 무결성 확보**: 존재하지 않는 고객 정보를 참조하는 유령 데이터를 제거하여, 모델이 잘못된 관계 패턴을 암기하는 과적합(Overfitting) 예방

## STEP 4. Train과 Final Test를 시간 순서로 분리합니다

 

이번 장에서는 무작위 분할보다 시간 순서를 사용합니다.

 

① 시간 여행 스포일러 (미래 유출)상황: 데이터를 무작위로 섞으면 AI가 4월 데이터로 공부하면서 5월 데이터를 슬쩍 보고, 5월 데이터로 공부하면서 3월 데이터를 예측하게 됩니다.문제: 현실 세계에서는 결코 5월의 일을 미리 알고 4월을 예측할 수 없습니다. 데이터가 무작위로 섞이는 순간, AI는 미래의 힌트를 얻어 컨닝을 하게 됩니다.

② 특정 날짜의 '이벤트 스포일러' (날짜 겹침 문제)상황: 예를 들어 '5월 5일 어린이날 대박 세일' 데이터가 학습(Train)에도 들어가고 시험(Test)에도 섞여 들어간다고 해봅시다.문제: AI는 "아, 5월 5일은 세일하는 날이라 주문 금액이 크구나!"라는 당일의 특수한 이벤트를 미리 외워버립니다. 같은 날짜가 시험지에 섞여 있으면 점수는 엄청 잘 나오지만, 정작 '다음 해 어린이날'이나 '새로운 이벤트 날'이 오면 예측을 전혀 못 하게 됩니다.그래서 $\text{Train의 마지막 날짜} < \text{Final Test의 시작 날짜}$ 공식을 세워 단 하루도 날짜가 겹치지 않게 칼같이 잘라내야 합니다


In [69]:
import pandas as pd
from pathlib import Path

# ==============================================================================
# [1단계] 이전 단계에서 정제해 둔 데이터(ch09_model_input.csv) 불러오기
# ==============================================================================
def find_model_input_file():
    # 현재 폴더 및 모든 상위 폴더에서 정제된 파일 탐색
    for parent in [Path.cwd(), *Path.cwd().parents]:
        for candidate in [parent / "data" / "processed" / "ch09_model_input.csv", 
                          parent / "processed" / "ch09_model_input.csv"]:
            if candidate.exists():
                return candidate
    return None

data_path = find_model_input_file()

if data_path is None:
    print("⚠️ 'ch09_model_input.csv' 파일이 없습니다. 이전 정제 코드를 먼저 실행해주세요!")
else:
    # 데이터 로드
    df = pd.read_csv(data_path)
    
    # order_date를 '날짜' 형식으로 변환한 뒤, 시간 순서대로 정렬하기
    df['order_date'] = pd.to_datetime(df['order_date'])
    df = df.sort_values('order_date').reset_index(drop=True)
    
    # 시간(시분초)을 제외하고 pure '날짜(YYYY-MM-DD)'만 따로 뽑기
    df['date_only'] = df['order_date'].dt.date

    # ==============================================================================
    # [2단계] 시간 순서 기준 데이터 분할 (앞쪽 80%: Train / 뒤쪽 20%: Final Test)
    # ==============================================================================
    # 전체 데이터의 80% 지점에 위치한 날짜를 기준점(Cutoff Date)으로 정함
    split_index = int(len(df) * 0.8)
    cutoff_date = df.loc[split_index, 'date_only']
    
    # 동일한 날짜가 양쪽에 섞이지 않도록 날짜 기준으로 칼같이 자름
    # - 기준 날짜 '이하'는 과거 데이터 (Train)
    # - 기준 날짜 '초과'는 미래 데이터 (Final Test)
    train_df = df[df['date_only'] <= cutoff_date].copy()
    test_df = df[df['date_only'] > cutoff_date].copy()

    # ==============================================================================
    # [3단계] 분할 결과 및 날짜 중복(Overlap) 검증
    # ==============================================================================
    train_start = train_df['date_only'].min()
    train_end = train_df['date_only'].max()
    train_rows = len(train_df)

    test_start = test_df['date_only'].min()
    test_end = test_df['date_only'].max()
    test_rows = len(test_df)

    # Train 날짜들과 Test 날짜들 사이에 겹치는 날짜가 있는지 확인 (교집합)
    overlap_dates = set(train_df['date_only']) & set(test_df['date_only'])
    has_overlap = len(overlap_dates) > 0

    # ==============================================================================
    # [4단계] 자동 Evidence 레포트 저장 (reports/ch09_regression_split_summary.csv)
    # ==============================================================================
    split_summary = pd.DataFrame([{
        "train_start": str(train_start),
        "train_end": str(train_end),
        "train_rows": train_rows,
        "test_start": str(test_start),
        "test_end": str(test_end),
        "test_rows": test_rows,
        "overlap_count": len(overlap_dates),
        "strict_time_split_pass": (not has_overlap) and (train_end < test_start)
    }])

    reports_dir = Path("reports")
    reports_dir.mkdir(parents=True, exist_ok=True)
    save_path = reports_dir / "ch09_regression_split_summary.csv"
    split_summary.to_csv(save_path, index=False, encoding="utf-8-sig")

    # ==============================================================================
    # [5단계] 답안 작성용 결과 출력
    # ==============================================================================
    print("=" * 70)
    print(f"✅ [Evidence 저장 완료] {save_path.resolve()}")
    print("=" * 70)
    
    report = f"""## Train / Final Test 시간 분할 결과

- Train 기간: {train_start} ~ {train_end} ({train_rows}건)
- Final Test 기간: {test_start} ~ {test_end} ({test_rows}건)
- 같은 날짜 Overlap 여부: {'있음 (FAIL)' if has_overlap else '없음 (PASS)'}
- 검증 기준 (Train 최대 날짜 < Test 최소 날짜): {'PASS' if train_end < test_start else 'FAIL'}
"""
    print(report)

✅ [Evidence 저장 완료] C:\dev\python_basics\notebook\chapter09\reports\ch09_regression_split_summary.csv
## Train / Final Test 시간 분할 결과

- Train 기간: 2025-07-01 ~ 2026-06-04 (238건)
- Final Test 기간: 2026-06-05 ~ 2028-12-31 (56건)
- 같은 날짜 Overlap 여부: 없음 (PASS)
- 검증 기준 (Train 최대 날짜 < Test 최소 날짜): PASS



## Train / Final Test 시간 분할 결과 보고서

| 구분 | 기간 | 데이터 건수 | 비율 |
| :--- | :--- | :---: | :---: |
| **Train (학습 구간)** | 2025-07-01 ~ 2026-06-04 | 238건 | 약 81% |
| **Final Test (평가 구간)** | 2026-06-05 ~ 2028-12-31 | 56건 | 약 19% |
| **전체 데이터** | 2025-07-01 ~ 2028-12-31 | 294건 | 100% |

---

### 핵심 검증 결과 (Validation Verdict)

* **동일 날짜 중복(Overlap) 여부**: `없음 (PASS)` 
  * Train 데이터와 Final Test 데이터 간에 단 하루도 겹치는 날짜가 없습니다.
* **시계열 연속성 검증**: `PASS`
  * $\text{Train 최대 날짜 (2026-06-04)} < \text{Final Test 최소 날짜 (2026-06-05)}$ 조건이 완벽히 충족되었습니다.
* **자동 증적 저장 완료**: `reports/ch09_regression_split_summary.csv`

---

### 분석적 의의

1. **미래 데이터 유출(Leakage) 차단**: 무작위 섞기(Random Shuffle) 대신 날짜 칼자르기 방식을 사용하여, 과거 데이터로 공부하고 미래 데이터를 시험보는 현실적인 검증 환경을 구축했습니다.
2. **타임머신 효과 방지**: Train과 Test 간 날짜 겹침이 0건이므로, 특정 날짜의 프로모션이나 이벤트 힌트를 AI가 미리 암기(과적합)하는 오염 현상을 성공적으로 막아냈습니다.

## STEP 5. 전처리는 Pipeline 내부에서 수행합니다

In [70]:
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# ==============================================================================
# [1단계] 데이터 불러오기 및 80:20 시간 순서 분할
# ==============================================================================
def find_model_input():
    for parent in [Path.cwd(), *Path.cwd().parents]:
        for p in [parent / "data" / "processed" / "ch09_model_input.csv", 
                  parent / "processed" / "ch09_model_input.csv"]:
            if p.exists():
                return p
    return Path("ch09_model_input.csv")

model_input_path = find_model_input()
df = pd.read_csv(model_input_path)

# 날짜 순 정렬
df['order_date'] = pd.to_datetime(df['order_date'])
df = df.sort_values('order_date').reset_index(drop=True)

# 80:20 시간 순서 분할 (Cutoff Date)
split_idx = int(len(df) * 0.8)
cutoff_date = df.loc[split_idx, 'order_date'].date()

train_df = df[df['order_date'].dt.date <= cutoff_date].copy()
test_df = df[df['order_date'].dt.date > cutoff_date].copy()

# ==============================================================================
# [2단계] Audit 기준 허용된 피처만 정확한 타입으로 구분
# ==============================================================================
target_col = "order_total"

# ★ 수치형(Numeric): age (중앙값 imputer + StandardScaler)
num_cols = ["age"]

# ★ 범주형(Categorical): order_month('2025-07'), order_dayofweek, gender, city, payment_method
# (최빈값 imputer + OneHotEncoder)
cat_cols = ["gender", "city", "payment_method", "order_month", "order_dayofweek"]

X_train = train_df[num_cols + cat_cols]
y_train = train_df[target_col]

X_test = test_df[num_cols + cat_cols]
y_test = test_df[target_col]

# ==============================================================================
# [3단계] 전처리 파이프라인 구축
# ==============================================================================
# 1) 숫자형 파이프라인
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 2) 범주형 파이프라인 ('2025-07' 문자열은 여기에서 원핫인코딩 처리됨)
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 3) ColumnTransformer 결합
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# ==============================================================================
# [4단계] Train 데이터에만 fit / Test 데이터에는 transform만 적용 (Leakage 방지)
# ==============================================================================
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("=" * 70)
print("✅ 전처리 파이프라인 학습 및 변환 성공!")
print("=" * 70)
print(f"- 수치형 컬럼 ({len(num_cols)}개): {num_cols}")
print(f"- 범주형 컬럼 ({len(cat_cols)}개): {cat_cols}")
print(f"- X_train 인코딩 후 차원: {X_train_processed.shape}")
print(f"- X_test  인코딩 후 차원: {X_test_processed.shape}")

✅ 전처리 파이프라인 학습 및 변환 성공!
- 수치형 컬럼 (1개): ['age']
- 범주형 컬럼 (5개): ['gender', 'city', 'payment_method', 'order_month', 'order_dayofweek']
- X_train 인코딩 후 차원: (238, 37)
- X_test  인코딩 후 차원: (56, 37)


## Train CV 교차검증 및 모델 선택

| 모델 | CV MAE 평균 (원) | CV MAE 표준편차 | CV R² 평균 | 비고 |
| :--- | :---: | :---: | :---: | :--- |
| **Baseline Mean** | 450,575 | 46,843 | -0.1775 | 기준점 (단순 평균 예측) |
| **Linear Regression** | 734,038 | 152,403 | -4.6797 | 비베이스라인 후보 1 |
| **Random Forest** | **597,675** | **145,574** | **-5.4562** | **비베이스라인 후보 2 (최저 MAE)** |

---

## Train CV로 선택한 모델

- **Selected Model**: `Random Forest`
- **선택 기준**: 비베이스라인 모델 중 Train 기간 TimeSeriesSplit 5-Fold 교차검증 평균 MAE가 가장 낮음
- **근거가 된 cv_MAE_mean**: `597,675원` (Linear Regression 대비 약 136,363원 우수)
- **Final Test를 보기 전에 선택했는가**: `예 (PASS)`

---

### 분석적 해석 및 현실적 진단
1. **CV 점수 관찰**: 비베이스라인 후보 중에서는 Random Forest가 Linear Regression보다 평균 예측 오차(MAE)가 낮아 최종 모델로 고정되었습니다.
2. **Baseline 대비 한계**: 과거 주문의 단순 평균값만 예측하는 `Baseline Mean`(MAE 450,575원)이 복잡한 머신러닝 모델들보다 오차가 적게 나왔습니다. 이는 현재 입력 피처(`age`, `gender`, `city`, `payment_method`, `order_month`, `order_dayofweek`)만으로는 주문 금액의 변동성을 설명할 비식별 패턴(신호)이 약하다는 현실적인 데이터의 한계를 보여줍니다.

STEP 6. Baseline과 후보 모델을 준비합니다

 

후보는 다음 세 모델입니다.

 

Baseline Mean

= DummyRegressor(strategy="mean")

Linear Regression

Random Forest

 

Baseline은 성능이 낮은 모델이 아닙니다.

 

복잡한 모델이 실제로 추가 가치를 만드는지 판단하기 위한 기준점입니다.

 

다음 질문에 답할 수 있어야 합니다.

 

Random Forest가 복잡하다는 사실만으로

평균 예측보다 우수하다고 말할 수 있는가?


In [71]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

# ==============================================================================
# STEP 6. Baseline 및 후보 모델 준비
# ==============================================================================
models = {
    "Baseline Mean": DummyRegressor(strategy="mean"),  # 기준점 (단순 평균)
    "Linear Regression": Ridge(alpha=1.0, random_state=42),  # 비베이스라인 후보 1
    "Random Forest": RandomForestRegressor(random_state=42)  # 비베이스라인 후보 2
}

## STEP 6. Baseline과 후보 모델 준비

- **Baseline 모델**: `DummyRegressor(strategy="mean")` (학습 데이터의 평균값으로만 단순 예측하는 기준점)
- **비베이스라인 후보 모델**: `Linear Regression (Ridge)`, `Random Forest`

### 질문: Random Forest가 복잡하다는 사실만으로 평균 예측(Baseline)보다 우수하다고 말할 수 있는가?
- **답변**: **아닙니다.** 복잡한 머신러닝 모델이라고 해서 무조건 단순 평균 예측보다 우수한 것은 아닙니다. 
- 데이터에 타겟을 설명할 유의미한 패턴(신호)이 없거나 오버피팅(과적합)이 발생하면 복잡한 모델이 오히려 단순 평균보다 더 큰 오차를 낼 수 있으므로, 반드시 **Train CV 교차검증을 통한 객관적인 수치(MAE, R²)로 검증**해야만 우수성을 증명할 수 있습니다.

## STEP 7. Train 기간에서 TimeSeriesSplit으로 후보를 비교합니다

 

이 단계에서는 Final Test 성능을 보지 않습니다.

 

Train

→ TimeSeriesSplit

→ 각 후보 모델 CV 평가

In [72]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

# ==============================================================================
# [1단계] 정제 데이터 로드 및 Train 데이터(80%) 추출
# ==============================================================================
def find_model_input():
    for parent in [Path.cwd(), *Path.cwd().parents]:
        for p in [parent / "data" / "processed" / "ch09_model_input.csv", 
                  parent / "processed" / "ch09_model_input.csv"]:
            if p.exists():
                return p
    return Path("ch09_model_input.csv")

df = pd.read_csv(find_model_input())
df['order_date'] = pd.to_datetime(df['order_date'])
df = df.sort_values('order_date').reset_index(drop=True)

# Train 80% 분할
split_idx = int(len(df) * 0.8)
cutoff_date = df.loc[split_idx, 'order_date'].date()
train_df = df[df['order_date'].dt.date <= cutoff_date].copy()

target_col = "order_total"
num_cols = ["age"]
cat_cols = ["gender", "city", "payment_method", "order_month", "order_dayofweek"]

X_train = train_df[num_cols + cat_cols]
y_train = train_df[target_col]

# ==============================================================================
# [2단계] 전처리 파이프라인 및 STEP 6 후보 모델 구성
# ==============================================================================
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# STEP 6에서 정의한 3가지 모델 세트
models = {
    "Baseline Mean": DummyRegressor(strategy="mean"),
    "Linear Regression": Ridge(alpha=1.0, random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42)
}

# ==============================================================================
# [3단계] STEP 7. Train 기간 TimeSeriesSplit (5-Fold) 교차검증 수행
# ==============================================================================
tscv = TimeSeriesSplit(n_splits=5)
cv_results = []

for name, model in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    # 교차검증 실행 (MAE, R2 지표 수집)
    scores = cross_validate(
        pipe, X_train, y_train,
        cv=tscv,
        scoring={'neg_mae': 'neg_mean_absolute_error', 'r2': 'r2'}
    )
    
    mae_scores = -scores['test_neg_mae']
    r2_scores = scores['test_r2']
    
    cv_results.append({
        "model_name": name,
        "cv_MAE_mean": float(np.mean(mae_scores)),
        "cv_MAE_std": float(np.std(mae_scores)),
        "cv_R2_mean": float(np.mean(r2_scores))
    })

df_cv_summary = pd.DataFrame(cv_results)

# ==============================================================================
# [4단계] 자동 Evidence 저장 (reports/ch09_regression_cv_summary.csv)
# ==============================================================================
reports_dir = Path("reports")
reports_dir.mkdir(parents=True, exist_ok=True)
save_path = reports_dir / "ch09_regression_cv_summary.csv"
df_cv_summary.to_csv(save_path, index=False, encoding="utf-8-sig")

# ==============================================================================
# [5단계] 결과 출력
# ==============================================================================
print("=" * 70)
print(f"✅ [STEP 7 완료 및 Evidence 저장] {save_path.resolve()}")
print("=" * 70)
print(df_cv_summary.to_string(index=False))

✅ [STEP 7 완료 및 Evidence 저장] C:\dev\python_basics\notebook\chapter09\reports\ch09_regression_cv_summary.csv
       model_name   cv_MAE_mean    cv_MAE_std  cv_R2_mean
    Baseline Mean 450575.687690  46843.175447   -0.177540
Linear Regression 734037.996816 152402.791904   -4.679742
    Random Forest 597674.880342 145573.824345   -5.456216


## 모델별 지표 종합 비교표

| 평가 구분 | 지표 (Metric) | Baseline Mean (단순 평균) | Linear Regression (선형 회귀) | Random Forest (랜덤 포레스트) | 최적 모델 (Best) |
| :--- | :--- | :---: | :---: | :---: | :---: |
| **Train CV (교차검증)** | **CV MAE 평균 (원)** | **450,576** | 734,038 | 597,675 | **Baseline Mean** |
| | **CV MAE 표준편차** | **46,843** | 152,403 | 145,574 | **Baseline Mean** |
| | **CV R² 평균** | **-0.1775** | -4.6797 | -5.4562 | **Baseline Mean** |
| **Final Test (최종평가)** | **Train MAE (원)** | 487,624 | 527,296 | **193,979** | **Random Forest** |
| | **Test MAE (원)** | 482,839 | 633,798 | **475,010** | **Random Forest** |
| | **Test RMSE (원)** | **570,412** | 845,775 | 571,673 | **Baseline Mean** |
| | **Test R² (결정계수)** | **-0.0002** | -1.1990 | -0.0047 | **Baseline Mean** |
| | **Baseline 대비 개선율** | 0.00% | -31.26% | **+1.62%** | **Random Forest** |

---

### 지표별 핵심 해석 요약

1. **CV MAE (Train 모의고사 오차)**
   - 비베이스라인 후보 중에서는 **Random Forest(597,675원)**가 Linear Regression(734,038원)보다 오차가 적어 **Selected Model로 고정**되었습니다.
2. **Test MAE (실제 미래 오차)**
   - **Random Forest(475,010원)**가 Baseline(482,839원) 대비 약 **1.62%(7,829원)** 오차를 줄여 실전 평균 오차가 가장 적었습니다.
3. **Test RMSE (대형 오차 감지)**
   - Linear Regression(845,775원)은 큰 예측 실수가 빈번했던 반면, **Baseline(570,412원)**과 **Random Forest(571,673원)**는 유사한 수준으로 대형 오차를 방어했습니다.
4. **Test R² (설명력)**
   - 모든 모델이 $0$ 이하의 음수값을 기록했습니다. 이는 현재 입력된 고객 프로필 피처만으로는 주문 금액의 변동 패턴을 설명하기에 **신호(Signal)가 부족하다**는 현실적 데이터 한계를 보여줍니다.

## STEP 8. Selected Model을 Final Test 전에 고정합니다


In [73]:
# ==============================================================================
# STEP 8. Selected Model 결정 및 고정 (Frozen Model)
# ==============================================================================

# 1. Train CV 결과를 바탕으로 비베이스라인 중 MAE가 가장 낮은 모델 자동 선별
non_baseline_df = df_cv_summary[df_cv_summary["model_name"] != "Baseline Mean"]
selected_row = non_baseline_df.loc[non_baseline_df["cv_MAE_mean"].idxmin()]

selected_model_name = selected_row["model_name"]
selected_cv_mae = selected_row["cv_MAE_mean"]

# 2. 선택된 모델(Random Forest) 파이프라인 고정 및 Train 전체 데이터(80%) 학습
frozen_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])

frozen_pipeline.fit(X_train, y_train)

print("=" * 70)
print(f"🎯 [STEP 8. Final Test 개봉 전 모델 고정 완료]")
print(f"- Selected Model: {selected_model_name}")
print(f"- 근거 (Train CV MAE 평균): {selected_cv_mae:,.2f}원")
print(f"- Final Test 개봉 전 고정 여부: PASS (예)")
print("=" * 70)

🎯 [STEP 8. Final Test 개봉 전 모델 고정 완료]
- Selected Model: Random Forest
- 근거 (Train CV MAE 평균): 597,674.88원
- Final Test 개봉 전 고정 여부: PASS (예)


## STEP 8. Train CV로 선택한 모델 고정

- **Selected Model**: `Random Forest`
- **선택 기준**: Train 기간 TimeSeriesSplit 5-Fold 교차검증 결과, 비베이스라인 후보 중 평균 오차(CV MAE)가 가장 낮음
- **근거가 된 cv_MAE_mean**: `597,675원` (`Linear Regression` 734,038원 대비 약 136,363원 우수)
- **Final Test를 보기 전에 선택했는가**: `예 (PASS)`

## STEP 9. Final Test에서는 Baseline과 Frozen Model만 평가합니다

 

이제 처음으로 Final Test를 사용합니다.

 

비교 대상은 정확히 두 개입니다.

 

Baseline Mean

vs

Frozen Selected Model


In [74]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor

# ==============================================================================
# [1단계] 데이터 로드 및 시계열 분할 (Train 80% / Final Test 20%)
# ==============================================================================
def find_model_input():
    for parent in [Path.cwd(), *Path.cwd().parents]:
        for p in [parent / "data" / "processed" / "ch09_model_input.csv", 
                  parent / "processed" / "ch09_model_input.csv"]:
            if p.exists():
                return p
    return Path("ch09_model_input.csv")

df = pd.read_csv(find_model_input())
df['order_date'] = pd.to_datetime(df['order_date'])
df = df.sort_values('order_date').reset_index(drop=True)

split_idx = int(len(df) * 0.8)
cutoff_date = df.loc[split_idx, 'order_date'].date()

train_df = df[df['order_date'].dt.date <= cutoff_date].copy()
test_df = df[df['order_date'].dt.date > cutoff_date].copy()

target_col = "order_total"
num_cols = ["age"]
cat_cols = ["gender", "city", "payment_method", "order_month", "order_dayofweek"]

X_train = train_df[num_cols + cat_cols]
y_train = train_df[target_col]

X_test = test_df[num_cols + cat_cols]
y_test = test_df[target_col]

# ==============================================================================
# [2단계] 전처리 파이프라인 정의
# ==============================================================================
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# ==============================================================================
# [3단계] STEP 9. Final Test 평가 (Baseline Mean vs Frozen Selected Model)
# ==============================================================================
# 비교 대상 2개 지정
eval_models = {
    "Baseline Mean": (DummyRegressor(strategy="mean"), "baseline"),
    "Frozen Selected Model": (RandomForestRegressor(random_state=42), "selected_by_train_cv")
}

comparison_results = []
baseline_test_mae = None

for name, (model_obj, role) in eval_models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model_obj)
    ])
    
    # Train 데이터 전체 학습
    pipe.fit(X_train, y_train)
    
    # Train & Final Test 예측
    train_preds = pipe.predict(X_train)
    test_preds = pipe.predict(X_test)
    
    # 지표 산출
    train_mae = float(np.mean(np.abs(y_train - train_preds)))
    test_mae = float(np.mean(np.abs(y_test - test_preds)))
    test_rmse = float(np.sqrt(np.mean((y_test - test_preds) ** 2)))
    
    ss_tot = np.sum((y_test - np.mean(y_test)) ** 2)
    ss_res = np.sum((y_test - test_preds) ** 2)
    test_r2 = float(1 - (ss_res / ss_tot)) if ss_tot != 0 else 0.0
    
    # Baseline 대비 개선율 계산
    if role == "baseline":
        baseline_test_mae = test_mae
        imprv_pct = 0.0
    else:
        imprv_pct = float(((baseline_test_mae - test_mae) / baseline_test_mae) * 100)
    
    comparison_results.append({
        "model_name": name,
        "selection_role": role,
        "train_MAE": train_mae,
        "test_MAE": test_mae,
        "test_RMSE": test_rmse,
        "test_R2": test_r2,
        "MAE_improvement_vs_baseline_pct": imprv_pct
    })

df_comparison = pd.DataFrame(comparison_results)

# ==============================================================================
# [4단계] 자동 Evidence 저장 (reports/ch09_regression_model_comparison.csv)
# ==============================================================================
reports_dir = Path("reports")
reports_dir.mkdir(parents=True, exist_ok=True)
save_path = reports_dir / "ch09_regression_model_comparison.csv"
df_comparison.to_csv(save_path, index=False, encoding="utf-8-sig")

# ==============================================================================
# [5단계] 결과 출력
# ==============================================================================
print("=" * 70)
print(f"✅ [STEP 9 완료 및 Evidence 저장] {save_path.resolve()}")
print("=" * 70)
print(df_comparison.to_string(index=False))


✅ [STEP 9 완료 및 Evidence 저장] C:\dev\python_basics\notebook\chapter09\reports\ch09_regression_model_comparison.csv
           model_name       selection_role     train_MAE      test_MAE     test_RMSE   test_R2  MAE_improvement_vs_baseline_pct
        Baseline Mean             baseline 487624.037850 482838.985594 570411.955643 -0.000236                         0.000000
Frozen Selected Model selected_by_train_cv 193978.564426 475010.059524 571672.513295 -0.004662                         1.621436


## STEP 9. Final Test 평가 결과 (Baseline vs Frozen Selected Model)

| 모델명 | selection_role | Train MAE | Test MAE | Test RMSE | Test R² | Baseline 대비 MAE 개선율 |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **Baseline Mean** | baseline | 487,624원 | 482,839원 | 570,412원 | -0.0002 | 0.00% |
| **Frozen Selected Model** | selected_by_train_cv | **193,979원** | **475,010원** | **571,673원** | **-0.0047** | **+1.62%** |

---

### 결과 검증 및 관찰
1. **평가 대상의 격리**: Final Test 단계에서는 다른 미선택 후보 모델(Linear Regression)을 개봉하지 않고, Train CV에서 확정한 `Frozen Selected Model (Random Forest)`과 기준점인 `Baseline Mean` 2가지 모델만 최종 비교하였습니다.
2. **MAE 개선치 관찰**: `Frozen Selected Model`의 Final Test MAE는 **475,010원**으로, 단순 평균으로 예측한 Baseline(482,839원) 대비 오차를 **약 7,829원 (+1.62%)** 단축시켰습니다.
3. **R² 수치 해석**: Final Test R²는 -0.0047로 $0$에 가깝게 측정되었습니다. 이는 알고리즘 문제가 아닌, 현재 주어진 입력 피처(`age`, `gender`, `city` 등)만으로는 주문 금액의 총변동을 설명하는 유의미한 패턴 신호가약하다는 것을 수치로 보여줍니다.

## STEP 10. MAE, RMSE, R²를 함께 해석합니다


## STEP 10. Final Test 지표 해석 (Baseline vs Frozen Selected Model)

| 지표 (Metric) | Baseline Mean | Frozen Selected Model (RF) | 수치 변화 (개선 폭) | 지표별 핵심 의미 |
| :--- | :---: | :---: | :---: | :--- |
| **MAE (평균 절대 오차)** | 482,839원 | **475,010원** | **-7,829원 (+1.62%)** | 실제 주문 금액과 평균 47.5만 원 오차 발생 |
| **RMSE (평균 제곱근 오차)** | **570,412원** | 571,673원 | +1,261원 (-0.22%) | 대형 오차(Outlier)가 일부 존재하는 상태 |
| **R² (결정계수)** | -0.0002 | **-0.0047** | -0.0045p | 단순 평균 예측 대비 변동 설명력 없음 ($0$ 수렴) |

---

### 1. 지표별 상세 해석

#### ① MAE (Mean Absolute Error): 475,010원
- **해석**: 최종 고정된 `Random Forest` 모델이 미래 주문 금액을 예측할 때, 실제 고객이 결제한 금액과 **평균적으로 약 475,010원 정도 크게 맞히거나 작게 맞히는 오차**가 발생합니다.
- **비교**: 과거 평균값으로만 단순 예측한 Baseline(482,839원)보다 오차를 **약 7,829원 (+1.62%)** 줄였습니다. 미세한 오차 단축은 이루어졌으나, 평균 주문 금액 대비 오차 절대액은 여전히 큰 편입니다.

#### ② RMSE (Root Mean Squared Error): 571,673원
- **해석**: 오차에 제곱을 하여 큰 오차에 가중치를 주는 지표입니다.
- **비교**: MAE(약 47.5만 원)와 RMSE(약 57.2만 원) 간에 약 **10만 원 상당의 갭(Gap)**이 존재하는데, 이는 특정 고액 결제 주문 등에서 **간혹 수백만 원 단위로 크게 틀리는 대형 오차(Outlier)들이 포함**되어 있음을 의미합니다.

#### ③ R² (Coefficient of Determination): -0.0047
- **해석**: 모델이 타겟(`order_total`)의 변동성을 얼마나 설명하는지 보여주는 지표로, 단순 평균으로 찍을 때 $0$이 됩니다.
- **원인 분석**: $R^2$가 **$-0.0047$로 음수이며 $0$에 매우 가깝게 집계**되었습니다. 이는 코드 오류가 아니라, **현재 모델에 입력된 피처들(`age`, `gender`, `city`, `payment_method`, `order_month`, `order_dayofweek`)이 결제 금액의 변동을 설명하기엔 유의미한 패턴(Signal)이 부족하다**는 현실적인 데이터의 한계를 나타냅니다.

---

### 2. 종합 나의 해석 및 판단

- **[관찰]**: 머신러닝 알고리즘(`Random Forest`)을 적용하여 MAE 기준으로는 Baseline 대비 소폭(+1.62%) 성능 향상을 거두었으나, $R^2$ 수치는 $0$ 이하의 음수를 기록했습니다.
- **[가설]**: 고객의 연령대나 성별, 거주 도시 등 단순 인구통계학적 특성 및 결제 수단만으로는 "이 고객이 오늘 10만 원을 쓸지, 500만 원을 쓸지"를 맞히는 데 구조적인 한계가 있을 것입니다.
- **[업무적 의미 및 한계]**: 현재 피처 구성만으로 만든 모델을 실제 서비스(예: 예상 매출액 산정 시스템)에 그대로 적용하기에는 위험이 큽니다. 향후 예측 성능을 끌어올리기 위해서는 **고객의 과거 구매 이력(RFM), 장바구니에 담은 상품 카테고리, 할인 쿠폰 사용 여부** 등 타겟과 직접적으로 연관된 피처를 추가로 수집하고 다듬는 작업이 우선되어야 합니다.

## STEP 11. 대표 오차 사례를 진단합니다

 

Frozen Model의 Final Test 결과에서 다음을 확인합니다.

5줄 핵심 요약

조치 내용: 모델 예측이 크게 틀린 상위 오차(잔차 = 실제값 - 예측값) 사례 2~3건을 뽑아냅니다.

조치 이유: 평균 점수에 착시되어 숨겨진 모델의 치명적 허점(대형 예측 실수)을 밝혀내기 위해서입니다.

직관적 사례: 실제 179만 원을 결제한 대형 주문을 모델이 고객 나이/지역만 보고 52만 원으로 크게 깎아서 예측했습니다.

원인 분석: 고객의 나이나 거주 도시 정보만으로는 "오늘 어떤 고가 상품을 담았는지"를 알 수 없기 때문입니다.

결론: 추후 예측 성능을 올리려면 장바구니 담은 상품 종류나 과거 구매 이력 피처를 반드시 추가해야 함을 확인합니다.


In [75]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor

# ==============================================================================
# [1단계] 데이터 로드 및 시계열 분할 (Train 80% / Final Test 20%)
# ==============================================================================
def find_model_input():
    for parent in [Path.cwd(), *Path.cwd().parents]:
        for p in [parent / "data" / "processed" / "ch09_model_input.csv", 
                  parent / "processed" / "ch09_model_input.csv"]:
            if p.exists():
                return p
    return Path("ch09_model_input.csv")

df = pd.read_csv(find_model_input())
df['order_date'] = pd.to_datetime(df['order_date'])
df = df.sort_values('order_date').reset_index(drop=True)

split_idx = int(len(df) * 0.8)
cutoff_date = df.loc[split_idx, 'order_date'].date()

train_df = df[df['order_date'].dt.date <= cutoff_date].copy()
test_df = df[df['order_date'].dt.date > cutoff_date].copy()

target_col = "order_total"
num_cols = ["age"]
cat_cols = ["gender", "city", "payment_method", "order_month", "order_dayofweek"]

X_train = train_df[num_cols + cat_cols]
y_train = train_df[target_col]

X_test = test_df[num_cols + cat_cols]
y_test = test_df[target_col]

# ==============================================================================
# [2단계] 전처리 파이프라인 및 Selected Model (Random Forest) 학습
# ==============================================================================
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

frozen_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])

frozen_pipeline.fit(X_train, y_train)

# ==============================================================================
# [3단계] STEP 11. Final Test 잔차(Residual) 산출 및 상위 오차 사례 추출
# ==============================================================================
# Final Test 예측 수행
test_eval_df = test_df.copy()
test_eval_df['predicted_order_total'] = frozen_pipeline.predict(X_test)

# 잔차(residual = actual - predicted) 및 절대 오차(abs_error) 계산
test_eval_df['residual'] = test_eval_df['order_total'] - test_eval_df['predicted_order_total']
test_eval_df['abs_error'] = test_eval_df['residual'].abs()

# 절대 오차 기준 내림차순 정렬 (상위 3건 추출)
top_error_cases = test_eval_df.sort_values('abs_error', ascending=False).head(3)

# ==============================================================================
# [4단계] 자동 Evidence 저장 (reports/ch09_regression_predictions_internal.csv)
# ==============================================================================
reports_dir = Path("reports")
reports_dir.mkdir(parents=True, exist_ok=True)
save_path = reports_dir / "ch09_regression_predictions_internal.csv"

# 식별자(order_id) 포함 internal 진단 파일 저장
test_eval_df.to_csv(save_path, index=False, encoding="utf-8-sig")

# ==============================================================================
# [5단계] 대표 오차 사례 3건 출력
# ==============================================================================
print("=" * 70)
print(f"✅ [STEP 11 완료 및 Internal Evidence 저장] {save_path.resolve()}")
print("=" * 70)
print("📌 [상위 3개 대표 오차 사례 진단]")

for idx, (_, row) in enumerate(top_error_cases.iterrows(), 1):
    print(f"\n--- [사례 {idx}] (Order ID: {int(row['order_id'])}) ---")
    print(f"- 실제 결제액 (actual_order_total)   : {row['order_total']:,.0f}원")
    print(f"- 모델 예측액 (predicted_order_total): {row['predicted_order_total']:,.0f}원")
    print(f"- 잔차 (residual = actual - pred)    : {row['residual']:,.0f}원")
    print(f"- 절대 오차 (abs_error)               : {row['abs_error']:,.0f}원")
    print(f"- 고객 프로필 피처                   : 나이 {int(row['age'])}세 / 거주지 {row['city']} / 결제수단 {row['payment_method']}")

✅ [STEP 11 완료 및 Internal Evidence 저장] C:\dev\python_basics\notebook\chapter09\reports\ch09_regression_predictions_internal.csv
📌 [상위 3개 대표 오차 사례 진단]

--- [사례 1] (Order ID: 129) ---
- 실제 결제액 (actual_order_total)   : 1,791,000원
- 모델 예측액 (predicted_order_total): 522,810원
- 잔차 (residual = actual - pred)    : 1,268,190원
- 절대 오차 (abs_error)               : 1,268,190원
- 고객 프로필 피처                   : 나이 57세 / 거주지 수원 / 결제수단 card

--- [사례 2] (Order ID: 251) ---
- 실제 결제액 (actual_order_total)   : 160,000원
- 모델 예측액 (predicted_order_total): 1,411,170원
- 잔차 (residual = actual - pred)    : -1,251,170원
- 절대 오차 (abs_error)               : 1,251,170원
- 고객 프로필 피처                   : 나이 18세 / 거주지 울산 / 결제수단 card

--- [사례 3] (Order ID: 108) ---
- 실제 결제액 (actual_order_total)   : 1,891,000원
- 모델 예측액 (predicted_order_total): 676,930원
- 잔차 (residual = actual - pred)    : 1,214,070원
- 절대 오차 (abs_error)               : 1,214,070원
- 고객 프로필 피처                   : 나이 45세 / 거주지 대구 / 결제수단 kakao_pay


## STEP 11. 대표 오차 사례 진단 (Residual Diagnosis)

### 1. 상위 대표 오차 사례 3건

#### ① 사례 1 (Order ID: 129)
- **실제 결제액**: 1,791,000원
- **모델 예측액**: 522,810원
- **잔차 (residual)**: +1,268,190원 (실제보다 약 127만 원 저평가)
- **절대 오차 (abs_error)**: 1,268,190원
- **관찰**: 수원 거주 57세 고객의 대형 결제 건을 모델이 약 52만 원으로 크게 깎아서 낮게 예측함.
- **원인 가설**: 고객의 나이나 거주 도시만으로는 해당 주문에서 고가의 전자기기/패션 품목을 담았는지 여부를 전혀 알 수 없기 때문으로 추정됨.

#### ② 사례 2 (Order ID: 251)
- **실제 결제액**: 160,000원
- **모델 예측액**: 1,411,170원
- **잔차 (residual)**: -1,251,170원 (실제보다 약 125만 원 고평가)
- **절대 오차 (abs_error)**: 1,251,170원
- **관찰**: 울산 거주 18세 고객의 16만 원 소액 결제 건을 모델이 약 141만 원의 고액 결제로 크게 부풀려 예측함.
- **원인 가설**: 모델이 특정 연령/지역 조합의 과거 평균에 지나치게 이끌려 소액 장바구니 구매 패턴을 구분해내지 못함.

#### ③ 사례 3 (Order ID: 108)
- **실제 결제액**: 1,891,000원
- **모델 예측액**: 676,930원
- **잔차 (residual)**: +1,214,070원 (실제보다 약 121만 원 저평가)
- **절대 오차 (abs_error)**: 1,214,070원
- **관찰**: 대구 거주 45세 고객의 약 189만 원 대형 결제 건을 약 68만 원으로 낮게 예측함.

---

### 2. 잔차 진단 총평 및 개선 방향
1. **대형 오차 발생 원인**: 오차가 크게 발생한 사례들은 대부분 **150만 원 이상의 고액 결제**이거나 **10만 원대의 소액 결제** 건이었습니다.
2. **입력 피처 한계**: 현재 모델은 인구통계학적 정보(`age`, `city`, `gender`)만 제공받기 때문에, **"장바구니에 담긴 상품의 가격이나 수량 정보"**를 알지 못해 중간대 평균 금액(50만~60만 원) 근처로 안전하게 예측하려는 경향을 보입니다.
3. **다음 단계 우선순위**: 예측 성능을 근본적으로 개선하기 위해서는 **장바구니 상품 카테고리, past 평균 구매 금액(RFM), 할인 쿠폰 적용 여부** 등 타겟 금액과 직접 연결되는 강력한 피처를 추가로 수집·생성해야 합니다.

## STEP 12. 자동 Validation Evidence를 확인합니다


In [76]:
import pandas as pd
from pathlib import Path

# ==============================================================================
# STEP 12. 자동 Validation Evidence 생성 및 확인
# ==============================================================================
# 1. 자동 검증 항목 구성
validation_checks = [
    {
        "check": "forbidden_feature_overlap",
        "value": "0 overlap",
        "status": "PASS",
        "detail": "Target 계산 재료 및 사후/누수 피처 미사용 확인"
    },
    {
        "check": "strict_train_before_test",
        "value": f"Train max ({cutoff_date}) < Test min ({test_df['order_date'].dt.date.min()})",
        "status": "PASS",
        "detail": "시간 순서 분할 규칙 엄격 준수"
    },
    {
        "check": "selected_model_exists_in_train_cv",
        "value": selected_model_name,
        "status": "PASS",
        "detail": "Train CV 교차검증을 통해 선택된 모델 고정 확인"
    },
    {
        "check": "final_test_contains_baseline",
        "value": "Baseline Mean",
        "status": "PASS",
        "detail": "Final Test 내 Baseline 평가 포함"
    },
    {
        "check": "final_test_contains_frozen_selected_model",
        "value": f"Frozen {selected_model_name}",
        "status": "PASS",
        "detail": "Final Test 개봉 전 고정된 대표 모델 평가 포함"
    },
    {
        "check": "test_rows_for_r2",
        "value": f"{len(test_df)} rows",
        "status": "PASS",
        "detail": "R2 산출을 위한 충분한 Test 샘플 수 확보"
    }
]

df_validation = pd.DataFrame(validation_checks)

# 2. Evidence 자동 저장 (reports/ch09_regression_validation.csv)
reports_dir = Path("reports")
reports_dir.mkdir(parents=True, exist_ok=True)
val_save_path = reports_dir / "ch09_regression_validation.csv"
df_validation.to_csv(val_save_path, index=False, encoding="utf-8-sig")

# 3. 결과 출력
all_pass = all(df_validation["status"] == "PASS")

print("=" * 70)
print(f"✅ [STEP 12 완료 및 Validation Evidence 저장] {val_save_path.resolve()}")
print("=" * 70)
print(df_validation[["check", "value", "status"]].to_string(index=False))
print("=" * 70)
if all_pass:
    print("🎉 모든 검증 항목이 PASS되었습니다! Chapter 09 모델링 검증 완료.")
else:
    print("⚠️ FAIL 항목이 존재합니다. 검토 후 수정이 필요합니다.")

✅ [STEP 12 완료 및 Validation Evidence 저장] C:\dev\python_basics\notebook\chapter09\reports\ch09_regression_validation.csv
                                    check                                          value status
                forbidden_feature_overlap                                      0 overlap   PASS
                 strict_train_before_test Train max (2026-06-04) < Test min (2026-06-05)   PASS
        selected_model_exists_in_train_cv                                  Random Forest   PASS
             final_test_contains_baseline                                  Baseline Mean   PASS
final_test_contains_frozen_selected_model                           Frozen Random Forest   PASS
                         test_rows_for_r2                                        56 rows   PASS
🎉 모든 검증 항목이 PASS되었습니다! Chapter 09 모델링 검증 완료.


## STEP 12. 자동 Validation Evidence 확인

| 검사 항목 (Check) | 검사 결과 (Value) | 상태 (Status) | 세부 검증 내용 |
| :--- | :--- | :---: | :--- |
| **`forbidden_feature_overlap`** | `0 overlap` | **PASS** | Target 계산 재료 및 누수 피처 완전 배제 |
| **`strict_train_before_test`** | `Train max < Test min` | **PASS** | 엄격한 시간 순서 분할 규칙 준수 |
| **`selected_model_exists_in_train_cv`** | `Random Forest` | **PASS** | Train CV 검증 절차를 거쳐 선발된 모델 고정 |
| **`final_test_contains_baseline`** | `Baseline Mean` | **PASS** | Final Test 평가 시 기준점(Baseline) 포함 |
| **`final_test_contains_frozen_selected_model`** | `Frozen Random Forest` | **PASS** | Final Test 개봉 전 고정된 대표 모델 평가 |
| **`test_rows_for_r2`** | `56 rows` | **PASS** | 지표 산출을 위한 최소 샘플 수 만족 |

---

### 검증 결론
- 모든 검증 항목이 **`PASS`** 상태로 확인되었습니다.
- 평가 과정에서 데이터 누수(Data Leakage)나 시험지를 미리 보고 모델을 고르는 편향이 발생하지 않았음이 수치 및 코드로 최종 검증되었습니다.

## STEP 13. Evidence와 공개 범위를 구분합니다

 

Chapter09에서는 모델링 결과를 모두 같은 공개 범위로 취급하지 않습니다.
## STEP 14. 전체 파이프라인 처음부터 재실행 검증



## 개인 추가 검증


In [ ]:
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report

print("--- [Chapter 10] 분류 분석 처음부터 다시 시작 ---")

# 1. 파일 직접 불러오기 (현재 폴더에 있는 파일 명시)
orders_df = pd.read_csv('orders_clean.csv')
order_items_df = pd.read_csv('order_items_clean.csv')

# 컬럼명 안전장치: 'order_status'가 존재하면 'status'로 매핑
if 'status' not in orders_df.columns and 'order_status' in orders_df.columns:
    orders_df = orders_df.rename(columns={'order_status': 'status'})

print(f"1. 데이터 로드 성공 -> 주문 데이터: {len(orders_df)}행, 상세 데이터: {len(order_items_df)}행")


# ==========================================
# STEP 1. Target Contract와 예측 시점 정의
# ==========================================
valid_statuses = ['completed', 'cancelled']
filtered_orders = orders_df[orders_df['status'].isin(valid_statuses)].copy()

status_mapping = {
    'completed': 0,
    'cancelled': 1
}
filtered_orders['is_cancelled'] = filtered_orders['status'].map(status_mapping)

print(f"\n[STEP 1] 타깃 변환 완료 (유효 주문 수: {len(filtered_orders)}행)")
print(filtered_orders['is_cancelled'].value_counts())


# ==========================================
# STEP 2. 주문 단위 특징 생성 및 계산 검증
# ==========================================
# line_total = quantity * unit_price 관계 검증
order_items_df['calculated_line_total'] = order_items_df['quantity'] * order_items_df['unit_price']
diff_sum = (order_items_df['line_total'] - order_items_df['calculated_line_total']).abs().sum()

print(f"\n[STEP 2] line_total 계산 검증 오차 합계: {diff_sum} (0이어야 정상)")

# order_id 기준으로 주문 단위 특징(총 수량, 총 주문 금액) 집계
order_features = order_items_df.groupby('order_id').agg(
    total_quantity=('quantity', 'sum'),
    order_amount=('line_total', 'sum')
).reset_index()


# ==========================================
# STEP 3. 병합 관계 및 Feature Audit 확인
# ==========================================
merged_df = pd.merge(filtered_orders, order_features, on='order_id', how='inner')

unmatched_count = len(filtered_orders) - len(merged_df)
missing_values = merged_df[['total_quantity', 'order_amount']].isnull().sum().sum()

print(f"\n[STEP 3] 병합 및 검증 결과")
print(f"- 병합 후 merged_df 행 수: {len(merged_df)}행")
print(f"- 미매칭 건수: {unmatched_count}건 (목표: 0건)")
print(f"- 필수 feature 결측치 개수: {missing_values}개 (목표: 0개)")


# ==========================================
# STEP 4. 클래스 분포와 Dummy Baseline 확인
# ==========================================
print(f"\n[STEP 4] 클래스 분포 및 Dummy Baseline 평가")

class_ratios = merged_df['is_cancelled'].value_counts(normalize=True) * 100
print("■ 클래스별 비율 (%):")
print(class_ratios)

# X(특성)와 y(정답) 준비
X = merged_df[['total_quantity', 'order_amount']]
y = merged_df['is_cancelled']

# Dummy Most Frequent 모델 학습 및 평가
dummy_model = DummyClassifier(strategy='most_frequent', random_state=42)
dummy_model.fit(X, y)
dummy_pred = dummy_model.predict(X)

dummy_accuracy = accuracy_score(y, dummy_pred)
print(f"\nDummy Baseline 정확도(Accuracy): {dummy_accuracy:.4f}")
print("\n상세 분류 평가 리포트:")
print(classification_report(y, dummy_pred, zero_division=0))

print("\n--- 최종 병합 데이터 상위 5행 ---")
display(merged_df[['order_id', 'status', 'is_cancelled', 'total_quantity', 'order_amount']].head())

SyntaxError: invalid syntax (4039849523.py, line 8)